# SCAFFOLD DESIGN

## Download new target

In [ ]:
# imports
from fragalysis.widgets import download
from pathlib import Path
import shutil
from os import environ
import json

In [ ]:
# this is the destination you should download to:
bulk_targets_dir = Path(environ["BULK"]) / "TARGETS"
print(bulk_targets_dir)

In [ ]:
# start the download widget
download()

## Initialise Directories for New Cycle

In [ ]:
target_name = "Target name"  # Change this to match the target name in Fragalysis.
cycle_number = 1  # Increment this for each new design cycle for this target.

In [ ]:

# all XChem-FFF work is kept under $HOME2/XChem-FFF; each target gets its own
# subdirectory here, and each design cycle its own subdirectory within that
xchem_fff_dir = Path(environ["HOME2"]) / "XChem-FFF"
target_dir = xchem_fff_dir / target_name.lower()
cycle_dir = target_dir / f"cycle_{cycle_number:02}"
fragmenstein_dir = cycle_dir / "fragmenstein"
knitwork_dir = cycle_dir / "knitwork"
knitwork_pure_output_dir = knitwork_dir / "knitwork_pure_output"
knitwork_impure_output_dir = knitwork_dir / "knitwork_impure_output"
gnina_dir = cycle_dir / "gnina"
gnina_inputs_dir = gnina_dir / "inputs"
gnina_inputs_fragmenstein_dir =  gnina_inputs_dir / "fragmenstein"
gnina_inputs_knitwork_pure_dir =  gnina_inputs_dir / "knitwork_pure"
gnina_inputs_knitwork_impure_dir =  gnina_inputs_dir / "knitwork_impure"
gnina_outputs_dir = gnina_dir / "outputs"
gnina_outputs_fragmenstein_dir =  gnina_outputs_dir / "fragmenstein"
gnina_outputs_knitwork_pure_dir =  gnina_outputs_dir / "knitwork_pure"
gnina_outputs_knitwork_impure_dir =  gnina_outputs_dir / "knitwork_impure"
moccassin_outputs_dir = cycle_dir / "moccassin_outputs"

In [ ]:
# create directories
cycle_dir.mkdir(parents=True, exist_ok=True)
fragmenstein_dir.mkdir(parents=True, exist_ok=True)
gnina_dir.mkdir(parents=True, exist_ok=True)
gnina_inputs_dir.mkdir(parents=True, exist_ok=True)
gnina_inputs_fragmenstein_dir.mkdir(parents=True, exist_ok=True)
gnina_inputs_knitwork_pure_dir.mkdir(parents=True, exist_ok=True)
gnina_inputs_knitwork_impure_dir.mkdir(parents=True, exist_ok=True)
gnina_outputs_dir.mkdir(parents=True, exist_ok=True)
gnina_outputs_fragmenstein_dir.mkdir(parents=True, exist_ok=True)
gnina_outputs_knitwork_pure_dir.mkdir(parents=True, exist_ok=True) 
gnina_outputs_knitwork_impure_dir.mkdir(parents=True, exist_ok=True) 
knitwork_dir.mkdir(parents=True, exist_ok=True)
knitwork_pure_output_dir.mkdir(parents=True, exist_ok=True)
knitwork_impure_output_dir.mkdir(parents=True, exist_ok=True)
moccassin_outputs_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# copy aligned files from $BULK to new directory
shutil.copytree(bulk_targets_dir / target_name / "aligned_files", target_dir / "aligned_files")

## Run Bulkdock / HIPPO

In [ ]:
%load_ext autoreload
%autoreload 2
import hippo
import mrich
from mrich import print
from pathlib import Path
from os import environ
import shutil
import molparse as mp
import plotly.express as px

In [ ]:
bulk_target_dir = Path(environ["BULK"]) / "TARGETS" / target_name

In [ ]:
!python -m bulkdock setup <TARGET_NAME> # Change <TARGET_NAME> to match the target name in Fragalysis.

This creates an SQLite in $BULK/TARGETS/<TARGET_NAME>


In [ ]:
animal = hippo.HIPPO(target_name, bulk_target_dir / f"{target_name}.sqlite")

## Create Fragmenstein and Knitwork Inputs

In [ ]:
# list available tags
animal.tags

In [ ]:
merge_tag = "[Other] FragScn" # change to your merge tag of choice, as tagged in Fragalysis
fragment_hits = animal.poses(tag=merge_tag)
fragment_hits.write_sdf(cycle_dir / "hits.sdf")

In [ ]:
# check how many tagged compounds will be processed
fragment_hits

In [ ]:
# download reference apo PDB for Fragmenstein - need a fragmenstein directory
ref_pose = fragment_hits[0]
shutil.copy(ref_pose.delig_path, cycle_dir / "fragmenstein")

## Run Fragmenstein

In [ ]:
print(fragmenstein_dir.resolve())

#### __In terminal :__

```bash
cd <fragmenstein_dir printed above>
sbatch --job-name "fragmenstein" --mem 16000 $HOME2/slurm/run_bash.sh $HOME2/slurm/run_fragmenstein.sh
```

## Run Knitwork on Squonk



__in Squonk terminal__  
```bash

python -m knitwork fragment ../hits.sdf
python -m knitwork pure-merge
python -m knitwork impure-merge
```

Output SDFs are written to `knitwork_pure_output/<target_name>_pure_merges.sdf` and
`knitwork_impure_output/<target_name>_impure_merges.sdf`. 

Transfer both output directories to the Knitwork directory for the current target and design cycle on IRIS
